<a href="https://colab.research.google.com/github/Fantiflex/Cognitive_biaises/blob/main/02_generate_social_matrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Purpose : Generate ONE shared set of stimuli for the social perception task.**
Produces two stimulus types from a single set of design parameters:


1.   SOCIAL matrices   — CFD face grids (Black / White / Men / Women)
2.   NON-SOCIAL matrices — dark-grey / light-grey filled circle grids used as a control condition, matched to critical trial prevalences

**Output structure** :
           

**Naming convention**
T<NNN>_<category>_<pct>pct.jpg
NNN      = zero-padded trial index (within stimulus type)
category = black | white | men | women | dark | light
pct      = true minority % (10 / 20 / 30 / 40 / 50)




**Design**



*Social:*     4 categories × 5 prevalence levels × N_REPS reps
* Critical : Black, White
* Filler   : Men, Women
*Non-social*: 2 categories × 5 prevalence levels × N_REPS reps
Control  : Dark (minority), Light (majority)
* same prevalence pool as social critical trials —
* dark circle ≡ minority face (Black); mirrors logic —

#           To scale up/down: ONLY edit N_REPS (and circle aesthetics if needed).
All counts, filenames, and the manifest update automatically.

In [15]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [16]:
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm")

print("base_dir exists:", base_dir.exists())
print("Contenu de base_dir:")
for p in base_dir.iterdir():
    print(p.name)

base_dir exists: True
Contenu de base_dir:
Images
CFD_codebook.xlsx
Untitled0.ipynb
outputs
minority_Faces
majority_Faces
01_select_cfd_images.ipynb
02_generate_matrices.ipynb
matrices


**Imports**

In [17]:
from pathlib import Path
import random
import shutil
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw

**Files loading & cleaning**

In [18]:
base_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm")

STIM_ROOT = base_dir

OUT_MIN = STIM_ROOT /"minority_Faces"
OUT_MAJ = STIM_ROOT /"majority_Faces"

SOCIAL_DIR = STIM_ROOT / "matrices" / "social"
NONSOCIAL_DIR = STIM_ROOT / "matrices" / "nonsocial"
MANIFEST_CSV = STIM_ROOT / "matrices" / "manifest.csv"

SOCIAL_DIR.mkdir(parents=True, exist_ok=True)
NONSOCIAL_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_MIN exists:", OUT_MIN.exists())
print("OUT_MAJ exists:", OUT_MAJ.exists())

print("Images minority:", len(list(OUT_MIN.rglob("*"))))
print("Images majority:", len(list(OUT_MAJ.rglob("*"))))

# Nettoyer les anciennes matrices si on relance le script
for folder in [SOCIAL_DIR, NONSOCIAL_DIR]:
    for old_file in folder.glob("*.jpg"):
        old_file.unlink()

OUT_MIN exists: True
OUT_MAJ exists: True
Images minority: 122
Images majority: 122


**Settings matrices**

In [19]:
GRID_COLS = 10
GRID_ROWS = 10
CELL_PX = 100
JPEG_QUAL = 82

PREVALENCE_LEVELS = [10, 20, 30, 40, 50]
N_REPS = 2
################# À ADAPTER AVEC LE CONTRASTE DE LA MATRICE SELECTIONNÉE ##############
CIRCLE_DARK_COL = "grey30"
CIRCLE_LIGHT_COL = "grey75"
CIRCLE_BG_COL = "white"
CIRCLE_RADIUS_PCT = 0.42

GLOBAL_SEED = 2022

n_faces = GRID_COLS * GRID_ROWS

N_SOCIAL_CATEGORIES = 4
N_NONSOCIAL_CATS = 2

N_TOTAL_SOCIAL = N_SOCIAL_CATEGORIES * len(PREVALENCE_LEVELS) * N_REPS
N_TOTAL_NONSOCIAL = N_NONSOCIAL_CATS * len(PREVALENCE_LEVELS) * N_REPS

# **Load faces images**

In [20]:

IMAGE_EXTENSIONS = {".jpg"}

def load_paths(folder, label):
    paths = [
        p for p in folder.rglob("*")
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if len(paths) == 0:
        raise FileNotFoundError(
            f"No images found in {folder}. "
            "Run the previous image selection/copy step first."
        )

    print(f"{label}: {len(paths)} images")
    return paths


minority_paths = load_paths(OUT_MIN, "Minority / Black")
majority_paths = load_paths(OUT_MAJ, "Majority / White")


def prep_face(path):
    img = Image.open(path).convert("RGB")

    width, height = img.size
    side = min(width, height)

    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side

    img = img.crop((left, top, right, bottom))
    img = img.resize((CELL_PX, CELL_PX))

    return img


minority_imgs = [prep_face(p) for p in minority_paths]
majority_imgs = [prep_face(p) for p in majority_paths]

print(f"{len(minority_imgs)} minority + {len(majority_imgs)} majority faces loaded")

Minority / Black: 122 images
Majority / White: 122 images
122 minority + 122 majority faces loaded


**Select faces w criteria**

In [21]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
############## SOCIAL TRIAL ################
social_trials = []

for question_category in ["Black", "White", "Men", "Women"]:
    for true_minority_pct in PREVALENCE_LEVELS:
        for rep in range(1, N_REPS + 1):
            social_trials.append({
                "question_category": question_category,
                "true_minority_pct": true_minority_pct,
                "stim_type": "social",
                "trial_type": "critical" if question_category in ["Black", "White"] else "filler",
                "true_majority_pct": 100 - true_minority_pct
            })

social_trials = pd.DataFrame(social_trials)
social_trials = social_trials.sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)
social_trials["trial_idx"] = np.arange(1, len(social_trials) + 1)

assert len(social_trials) == N_TOTAL_SOCIAL

**Functions to ASSEMBLE MATRICES**

In [22]:
def assemble_grid(cells):
    matrix = Image.new("RGB", (GRID_COLS * CELL_PX, GRID_ROWS * CELL_PX))

    for idx, cell in enumerate(cells):
        row = idx // GRID_COLS
        col = idx % GRID_COLS

        x = col * CELL_PX
        y = row * CELL_PX

        matrix.paste(cell, (x, y))

    return matrix


def make_face_matrix(n_minority, img_seed):
    rng = random.Random(img_seed)

    n_majority = n_faces - n_minority

    if n_minority > len(minority_imgs):
        raise ValueError(f"Not enough minority images: need {n_minority}, have {len(minority_imgs)}")

    if n_majority > len(majority_imgs):
        raise ValueError(f"Not enough majority images: need {n_majority}, have {len(majority_imgs)}")

    cells = (
        rng.sample(minority_imgs, n_minority) +
        rng.sample(majority_imgs, n_majority)
    )

    rng.shuffle(cells)

    return assemble_grid(cells)


# **Generate the actual matrices**

In [23]:
social_manifest = []

print(f"Generating {N_TOTAL_SOCIAL} social matrices...")

for i, row in social_trials.iterrows():
    trial_idx = int(row["trial_idx"])
    category = row["question_category"]
    true_minority_pct = int(row["true_minority_pct"])

    fname = f"T{trial_idx:03d}_{category.lower()}_{true_minority_pct}pct.jpg"
    fpath = SOCIAL_DIR / fname

    img_seed = GLOBAL_SEED + trial_idx * 13
    n_min = round(n_faces * true_minority_pct / 100)

    img = make_face_matrix(n_min, img_seed)
    img.save(fpath, format="JPEG", quality=JPEG_QUAL)

    social_manifest.append({
        "stim_type": "social",
        "trial_idx": trial_idx,
        "filename": fname,
        "url_stub": f"social/{fname}",
        "trial_type": row["trial_type"],
        "category": category,
        "true_minority_pct": true_minority_pct,
        "true_majority_pct": int(row["true_majority_pct"]),
        "img_seed": img_seed
    })

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{N_TOTAL_SOCIAL} done")

Generating 40 social matrices...
10/40 done
20/40 done
30/40 done
40/40 done
